# Robust Multivariate Statistical Tests: Nighttime Recovery, VPD, and Daytime Heat

## Scientific Background

This notebook applies **rigorous statistical testing** to evaluate the relationships between:
1. **Daytime Heat Exposure** (8am-8pm)
2. **Nighttime Recovery** (8pm-6am)
3. **Vapor Pressure Deficit (VPD)**
4. **Cattle Mortality**

### Statistical Framework

We employ multiple complementary approaches:

**1. Bivariate Tests:**
- Pearson correlation (linear relationships)
- Spearman correlation (monotonic relationships)
- Kendall's tau (ordinal relationships)

**2. Partial Correlation:**
- Control for confounding variables
- Isolate independent effects

**3. Multiple Regression:**
- Assess simultaneous effects
- Test interaction terms
- Variance inflation factor (VIF) for multicollinearity

**4. ANOVA/MANOVA:**
- Compare categorical groups
- Multi-factor analysis

**5. Principal Component Analysis (PCA):**
- Data-driven dimensionality reduction
- Identify latent factors

**6. Granger Causality:**
- Test temporal precedence
- Identify lead-lag relationships

### Research Questions

1. Which stress factor (daytime heat, nighttime recovery, VPD) has the strongest independent effect on mortality?
2. Are effects additive or interactive (synergistic)?
3. Do partial correlations (controlling for confounders) change interpretations?
4. Can we identify latent stress factors through PCA?
5. Do heat metrics Granger-cause mortality (temporal precedence)?

### Hypotheses

**H1**: Daytime heat has the strongest partial correlation after controlling for other factors  
**H2**: Nighttime recovery and daytime heat interact synergistically  
**H3**: VPD effects remain significant after controlling for temperature  
**H4**: PCA reveals a single dominant "heat stress" factor  
**H5**: Daytime heat Granger-causes mortality at 1-2 week lags

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr, kendalltau, chi2, f
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Import project configuration
import sys
sys.path.append('../../')
from config import (
    STATE_NAMES, STATE_ABBRS, STATE_REGIONS,
    FOCUS_STATES, CATTLE_REGIONS, CUSTOM_REGIONS, SEASONS,
    CATTLE_DATA_DIR
)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully")
print("Using sklearn-based statistical methods for maximum compatibility")

## 1. Data Loading and Preparation

In [ ]:
# Load and merge cattle-heat dataset
import xarray as xr
from config import PROCESSED_WEEKLY_DIR, MASK_FILE, CATTLE_DATA_FILE, CATTLE_REGION_IDS

print("Loading climate data...")
ds_night = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')
ds_day = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')
ds_vpd = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_vpd.nc')

# Get week dates from climate data
week_dates = ds_night['week'].values
n_weeks = len(week_dates)

print(f"Climate data: {n_weeks} weeks")
print(f"  From: {pd.to_datetime(week_dates[0]).date()}")
print(f"  To: {pd.to_datetime(week_dates[-1]).date()}")

# Load state mask for regional aggregation
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask.load()
ds_mask.close()

# Helper function to compute regional mean
def compute_regional_mean(data, state_ids, state_mask):
    """Compute spatial mean across multiple states."""
    combined_mask = xr.zeros_like(state_mask, dtype=bool)
    for state_id in state_ids:
        combined_mask = combined_mask | (state_mask == state_id)
    masked_data = data.where(combined_mask)
    return masked_data.mean(dim=['lat', 'lon']).astype(np.float64)

# Define region IDs
region_4_ids = CATTLE_REGION_IDS['region_4']
region_6_ids = CATTLE_REGION_IDS['region_6']

# Compute regional climate metrics for Region 4
print("\nComputing regional climate metrics...")
r4_metrics = pd.DataFrame({
    'week_ending': pd.to_datetime(week_dates),
    'region': 4,
    'mean_daytime_hours_above_30': compute_regional_mean(ds_day['hours_above_30'], region_4_ids, state_mask).values,
    'mean_daytime_hours_above_35': compute_regional_mean(ds_day['hours_above_35'], region_4_ids, state_mask).values,
    'mean_nighttime_hours_above_21': compute_regional_mean(ds_night['hours_above_21'], region_4_ids, state_mask).values,
    'mean_nighttime_hours_above_24': compute_regional_mean(ds_night['hours_above_24'], region_4_ids, state_mask).values,
    'mean_vpd_mean': compute_regional_mean(ds_vpd['vpd_mean'], region_4_ids, state_mask).values,
    'mean_vpd_max': compute_regional_mean(ds_vpd['vpd_max'], region_4_ids, state_mask).values,
})

# Compute regional climate metrics for Region 6
r6_metrics = pd.DataFrame({
    'week_ending': pd.to_datetime(week_dates),
    'region': 6,
    'mean_daytime_hours_above_30': compute_regional_mean(ds_day['hours_above_30'], region_6_ids, state_mask).values,
    'mean_daytime_hours_above_35': compute_regional_mean(ds_day['hours_above_35'], region_6_ids, state_mask).values,
    'mean_nighttime_hours_above_21': compute_regional_mean(ds_night['hours_above_21'], region_6_ids, state_mask).values,
    'mean_nighttime_hours_above_24': compute_regional_mean(ds_night['hours_above_24'], region_6_ids, state_mask).values,
    'mean_vpd_mean': compute_regional_mean(ds_vpd['vpd_mean'], region_6_ids, state_mask).values,
    'mean_vpd_max': compute_regional_mean(ds_vpd['vpd_max'], region_6_ids, state_mask).values,
})

# Combine regions
climate_data = pd.concat([r4_metrics, r6_metrics], ignore_index=True)

# Load cattle data
print("Loading cattle data...")
cattle_data = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])
cattle_data = cattle_data.rename(columns={'date': 'week_ending'})

# Reshape cattle data for regions 4 and 6
cattle_r4 = cattle_data[['week_ending', 'region_4_beef_dairy']].copy()
cattle_r4['region'] = 4
cattle_r4['total_beef_dairy'] = cattle_r4['region_4_beef_dairy']
cattle_r4 = cattle_r4.drop(columns=['region_4_beef_dairy'])

cattle_r6 = cattle_data[['week_ending', 'region_6_beef_dairy']].copy()
cattle_r6['region'] = 6
cattle_r6['total_beef_dairy'] = cattle_r6['region_6_beef_dairy']
cattle_r6 = cattle_r6.drop(columns=['region_6_beef_dairy'])

cattle_regional = pd.concat([cattle_r4, cattle_r6], ignore_index=True)

# Merge cattle and climate data
print("Merging datasets...")
cattle_heat = pd.merge(cattle_regional, climate_data, on=['week_ending', 'region'], how='inner')

print(f"\nMerged dataset: {len(cattle_heat)} region-weeks")
print(f"Date range: {cattle_heat['week_ending'].min()} to {cattle_heat['week_ending'].max()}")

# Add temporal features
cattle_heat['year'] = cattle_heat['week_ending'].dt.year
cattle_heat['month'] = cattle_heat['week_ending'].dt.month

def get_season(month):
    if month in SEASONS['Winter']:
        return 'Winter'
    elif month in SEASONS['Spring']:
        return 'Spring'
    elif month in SEASONS['Summer']:
        return 'Summer'
    else:
        return 'Fall'

cattle_heat['season'] = cattle_heat['month'].apply(get_season)

# Focus on Regions 4 and 6
cattle_focus = cattle_heat[cattle_heat['region'].isin([4, 6])].copy()

print(f"\nFocused on Regions 4 & 6: {len(cattle_focus)} region-weeks")

In [ ]:
# Define key variables for multivariate analysis
mortality_var = 'total_beef_dairy'

# Core heat stress variables
core_variables = {
    'Daytime Heat (30°C)': 'mean_daytime_hours_above_30',
    'Daytime Heat (35°C)': 'mean_daytime_hours_above_35',
    'Nighttime Recovery (21°C)': 'mean_nighttime_hours_above_21',
    'Nighttime Recovery (24°C)': 'mean_nighttime_hours_above_24',
    'VPD Mean': 'mean_vpd_mean',
    'VPD Max': 'mean_vpd_max'
}

# Create clean dataset for statistical analysis
analysis_vars = list(core_variables.values()) + [mortality_var, 'region', 'season', 'year']
stats_data = cattle_focus[analysis_vars].dropna().copy()

print(f"\nClean dataset for statistical analysis: {len(stats_data)} observations")
print(f"Variables: {len(core_variables)} heat stress metrics + mortality")
print(f"\nDescriptive statistics:")
print(stats_data[list(core_variables.values()) + [mortality_var]].describe())

## 2. Comprehensive Correlation Matrix

Three correlation methods: Pearson, Spearman, and Kendall's tau.

In [ ]:
# Calculate correlation matrices
import os
os.makedirs('../../figures/multivariate_statistical_tests', exist_ok=True)

# Pearson (linear)
pearson_matrix = stats_data[list(core_variables.values()) + [mortality_var]].corr(method='pearson')

# Spearman (monotonic)
spearman_matrix = stats_data[list(core_variables.values()) + [mortality_var]].corr(method='spearman')

# Kendall (ordinal)
kendall_matrix = stats_data[list(core_variables.values()) + [mortality_var]].corr(method='kendall')

print("="*80)
print("CORRELATION MATRICES")
print("="*80)
print("\nPearson Correlation with Mortality:")
print(pearson_matrix[mortality_var].sort_values(ascending=False))
print("\nSpearman Correlation with Mortality:")
print(spearman_matrix[mortality_var].sort_values(ascending=False))
print("\nKendall Tau Correlation with Mortality:")
print(kendall_matrix[mortality_var].sort_values(ascending=False))

In [ ]:
# Visualize correlation matrices
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Create readable labels
readable_labels = list(core_variables.keys()) + ['Mortality']

# Plot 1: Pearson
sns.heatmap(pearson_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[0], cbar_kws={'label': 'Pearson r'},
            xticklabels=readable_labels, yticklabels=readable_labels)
axes[0].set_title('Pearson Correlation\n(Linear Relationships)', fontsize=12, fontweight='bold')

# Plot 2: Spearman
sns.heatmap(spearman_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[1], cbar_kws={'label': 'Spearman ρ'},
            xticklabels=readable_labels, yticklabels=readable_labels)
axes[1].set_title('Spearman Correlation\n(Monotonic Relationships)', fontsize=12, fontweight='bold')

# Plot 3: Kendall
sns.heatmap(kendall_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[2], cbar_kws={'label': 'Kendall τ'},
            xticklabels=readable_labels, yticklabels=readable_labels)
axes[2].set_title('Kendall Tau Correlation\n(Ordinal Relationships)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../../figures/multivariate_statistical_tests/01_correlation_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 01_correlation_matrices.png")

## 3. Partial Correlation Analysis

Control for confounding variables to isolate independent effects.

In [ ]:
# Compute partial correlations using sklearn
from scipy.stats import pearsonr

def partial_correlation(data, x, y, controls):
    """
    Calculate partial correlation between x and y, controlling for variables in controls.
    
    Uses residualization method with sklearn:
    1. Regress x on controls, get residuals
    2. Regress y on controls, get residuals
    3. Correlate the two sets of residuals
    """
    # Prepare control variables
    X_controls = data[controls].values
    
    # Regress x on controls using sklearn
    model_x = LinearRegression()
    model_x.fit(X_controls, data[x])
    residuals_x = data[x] - model_x.predict(X_controls)
    
    # Regress y on controls using sklearn
    model_y = LinearRegression()
    model_y.fit(X_controls, data[y])
    residuals_y = data[y] - model_y.predict(X_controls)
    
    # Correlate residuals
    r, p = pearsonr(residuals_x, residuals_y)
    
    return r, p

# Calculate partial correlations
print("="*80)
print("PARTIAL CORRELATION ANALYSIS")
print("="*80)

partial_results = []

# Test each heat variable controlling for others
heat_vars = list(core_variables.values())

for var_name, var in core_variables.items():
    # Control variables = all other heat metrics
    controls = [v for v in heat_vars if v != var]
    
    # Calculate partial correlation
    r_partial, p_partial = partial_correlation(stats_data, var, mortality_var, controls)
    
    # Also get simple correlation for comparison
    r_simple, p_simple = pearsonr(stats_data[var], stats_data[mortality_var])
    
    partial_results.append({
        'Variable': var_name,
        'Simple_r': r_simple,
        'Simple_p': p_simple,
        'Partial_r': r_partial,
        'Partial_p': p_partial,
        'Difference': r_partial - r_simple,
        'Controls': ', '.join([k for k, v in core_variables.items() if v in controls])
    })
    
    print(f"\n{var_name}:")
    print(f"  Simple correlation: r = {r_simple:.4f}, p = {p_simple:.4e}")
    print(f"  Partial correlation: r = {r_partial:.4f}, p = {p_partial:.4e}")
    print(f"  Controlling for: {len(controls)} variables")
    print(f"  Change in r: {r_partial - r_simple:+.4f}")

partial_df = pd.DataFrame(partial_results)

print("\n" + "="*80)
print("SUMMARY: Simple vs Partial Correlations")
print("="*80)
print(partial_df[['Variable', 'Simple_r', 'Partial_r', 'Difference']].to_string(index=False))

In [ ]:
# Visualize partial correlations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Simple vs Partial comparison
ax = axes[0]
x = np.arange(len(partial_df))
width = 0.35

ax.bar(x - width/2, partial_df['Simple_r'], width, label='Simple', color='steelblue', alpha=0.8)
ax.bar(x + width/2, partial_df['Partial_r'], width, label='Partial', color='coral', alpha=0.8)

ax.set_xlabel('Variable', fontsize=11, fontweight='bold')
ax.set_ylabel('Correlation with Mortality', fontsize=11, fontweight='bold')
ax.set_title('Simple vs Partial Correlations\nControlling for Other Heat Metrics', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(partial_df['Variable'], rotation=45, ha='right', fontsize=9)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Change in correlation
ax = axes[1]
colors = ['green' if x > 0 else 'red' for x in partial_df['Difference']]
ax.barh(range(len(partial_df)), partial_df['Difference'], color=colors, alpha=0.7)

ax.set_yticks(range(len(partial_df)))
ax.set_yticklabels(partial_df['Variable'], fontsize=9)
ax.set_xlabel('Change in Correlation\n(Partial - Simple)', fontsize=11, fontweight='bold')
ax.set_title('Effect of Controlling for Confounders', fontsize=12, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/multivariate_statistical_tests/02_partial_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 02_partial_correlations.png")

## 4. Multiple Regression with Interaction Terms

In [ ]:
# Prepare data for regression
# Rename columns for formula syntax
reg_data = stats_data.copy()
reg_data.columns = [col.replace('mean_', '').replace('_', ' ').replace('above', '>').replace('hours', 'hrs') 
                    for col in reg_data.columns]

# Simplify column names further
col_map = {
    'daytime hrs > 30': 'heat30',
    'daytime hrs > 35': 'heat35',
    'nighttime hrs > 21': 'night21',
    'nighttime hrs > 24': 'night24',
    'vpd mean': 'vpd_mean',
    'vpd max': 'vpd_max',
    'total beef dairy': 'mortality'
}

for old_name, new_name in col_map.items():
    if old_name in reg_data.columns:
        reg_data.rename(columns={old_name: new_name}, inplace=True)

print("Data prepared for regression")
print(f"Columns: {reg_data.columns.tolist()}")

In [ ]:
# Model 1: Additive effects only (using sklearn)
print("="*80)
print("MODEL 1: ADDITIVE EFFECTS")
print("="*80)

# Prepare features
X_features = ['heat30', 'heat35', 'night21', 'vpd_mean']

# Add dummy variables for region and season
region_dummies = pd.get_dummies(reg_data['region'], prefix='region', drop_first=True)
season_dummies = pd.get_dummies(reg_data['season'], prefix='season', drop_first=True)

X_model1 = pd.concat([
    reg_data[X_features],
    region_dummies,
    season_dummies
], axis=1)

y_model1 = reg_data['mortality']

# Fit model
model1 = LinearRegression()
model1.fit(X_model1, y_model1)

# Predictions
y_pred1 = model1.predict(X_model1)

# Calculate R²
from sklearn.metrics import r2_score, mean_squared_error
r2_model1 = r2_score(y_model1, y_pred1)
mse_model1 = mean_squared_error(y_model1, y_pred1)
rmse_model1 = np.sqrt(mse_model1)

# Adjusted R²
n = len(y_model1)
p = X_model1.shape[1]
adj_r2_model1 = 1 - (1 - r2_model1) * (n - 1) / (n - p - 1)

print(f"\nModel Performance:")
print(f"  R² = {r2_model1:.4f}")
print(f"  Adjusted R² = {adj_r2_model1:.4f}")
print(f"  RMSE = {rmse_model1:.2f}")
print(f"  N observations = {n}")
print(f"  N parameters = {p}")

# Calculate VIF manually for multicollinearity check
print("\n" + "="*80)
print("MULTICOLLINEARITY CHECK (VIF)")
print("="*80)
print("VIF < 5: Low multicollinearity")
print("VIF 5-10: Moderate multicollinearity")
print("VIF > 10: High multicollinearity (problematic)")
print("")

def calculate_vif(X):
    """Calculate VIF for each feature"""
    vif_data = pd.DataFrame()
    vif_data['Variable'] = X.columns
    vif_values = []
    
    for i in range(len(X.columns)):
        # For each variable, regress it on all others
        y_var = X.iloc[:, i]
        X_others = X.drop(X.columns[i], axis=1)
        
        model = LinearRegression()
        model.fit(X_others, y_var)
        r2 = r2_score(y_var, model.predict(X_others))
        
        # VIF = 1 / (1 - R²)
        vif = 1 / (1 - r2) if r2 < 0.9999 else float('inf')
        vif_values.append(vif)
    
    vif_data['VIF'] = vif_values
    return vif_data

# Calculate VIF for core variables only
X_vif = reg_data[['heat30', 'heat35', 'night21', 'vpd_mean']].dropna()
vif_data = calculate_vif(X_vif)
print(vif_data.to_string(index=False))

# Print coefficients
print("\n" + "="*80)
print("MODEL COEFFICIENTS")
print("="*80)
coef_df = pd.DataFrame({
    'Feature': X_model1.columns,
    'Coefficient': model1.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))

In [ ]:
# Model 2: Include key interactions
print("\n" + "="*80)
print("MODEL 2: WITH INTERACTION TERMS")
print("="*80)

# Create interaction terms
reg_data['heat30_x_vpd'] = reg_data['heat30'] * reg_data['vpd_mean']
reg_data['heat30_x_night21'] = reg_data['heat30'] * reg_data['night21']

# Prepare features with interactions
X_features_interact = ['heat30', 'heat35', 'night21', 'vpd_mean', 
                       'heat30_x_vpd', 'heat30_x_night21']

X_model2 = pd.concat([
    reg_data[X_features_interact],
    region_dummies,
    season_dummies
], axis=1)

y_model2 = reg_data['mortality']

# Fit model
model2 = LinearRegression()
model2.fit(X_model2, y_model2)

# Predictions
y_pred2 = model2.predict(X_model2)

# Calculate R²
r2_model2 = r2_score(y_model2, y_pred2)
mse_model2 = mean_squared_error(y_model2, y_pred2)
rmse_model2 = np.sqrt(mse_model2)

# Adjusted R²
n2 = len(y_model2)
p2 = X_model2.shape[1]
adj_r2_model2 = 1 - (1 - r2_model2) * (n2 - 1) / (n2 - p2 - 1)

print(f"\nModel Performance:")
print(f"  R² = {r2_model2:.4f}")
print(f"  Adjusted R² = {adj_r2_model2:.4f}")
print(f"  RMSE = {rmse_model2:.2f}")
print(f"  N observations = {n2}")
print(f"  N parameters = {p2}")

# Print interaction coefficients
print("\n" + "="*80)
print("INTERACTION TERM COEFFICIENTS")
print("="*80)
interaction_coefs = pd.DataFrame({
    'Feature': X_model2.columns,
    'Coefficient': model2.coef_
})
# Show only interaction terms
interaction_terms = interaction_coefs[interaction_coefs['Feature'].str.contains('_x_')]
print(interaction_terms.to_string(index=False))

# Compare models
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(f"Model 1 (Additive): R² = {r2_model1:.4f}, Adj R² = {adj_r2_model1:.4f}, RMSE = {rmse_model1:.2f}")
print(f"Model 2 (Interactions): R² = {r2_model2:.4f}, Adj R² = {adj_r2_model2:.4f}, RMSE = {rmse_model2:.2f}")

# F-test for nested models
print("\n" + "="*80)
print("F-TEST FOR MODEL IMPROVEMENT")
print("="*80)

# Calculate F-statistic
rss1 = np.sum((y_model1 - y_pred1) ** 2)
rss2 = np.sum((y_model2 - y_pred2) ** 2)
df1 = p
df2 = p2
n_obs = n

f_stat = ((rss1 - rss2) / (df2 - df1)) / (rss2 / (n_obs - df2 - 1))
p_value = 1 - stats.f.cdf(f_stat, df2 - df1, n_obs - df2 - 1)

print(f"  F-statistic: {f_stat:.3f}")
print(f"  df1: {df2 - df1} (additional parameters)")
print(f"  df2: {n_obs - df2 - 1} (residual)")
print(f"  P-value: {p_value:.4e}")
print(f"  Conclusion: Interaction model is {'significantly' if p_value < 0.05 else 'not significantly'} better")

# R² improvement
r2_improvement = r2_model2 - r2_model1
adj_r2_improvement = adj_r2_model2 - adj_r2_model1

print(f"\nR² Improvement:")
print(f"  ΔR² = {r2_improvement:+.4f}")
print(f"  ΔAdj R² = {adj_r2_improvement:+.4f}")
print(f"  Relative improvement = {(r2_improvement/r2_model1)*100:+.2f}%")

## 5. Principal Component Analysis

In [ ]:
# Perform PCA on heat stress variables
pca_vars = ['heat30', 'heat35', 'night21', 'vpd_mean']
pca_data = reg_data[pca_vars].dropna()

# Standardize
scaler = StandardScaler()
pca_scaled = scaler.fit_transform(pca_data)

# Fit PCA
pca = PCA()
pca_components = pca.fit_transform(pca_scaled)

print("="*80)
print("PRINCIPAL COMPONENT ANALYSIS")
print("="*80)
print(f"\nExplained Variance Ratio:")
for i, var in enumerate(pca.explained_variance_ratio_, 1):
    print(f"  PC{i}: {var:.4f} ({var*100:.2f}%)")

print(f"\nCumulative Variance Explained:")
cumsum = np.cumsum(pca.explained_variance_ratio_)
for i, var in enumerate(cumsum, 1):
    print(f"  First {i} PCs: {var:.4f} ({var*100:.2f}%)")

print(f"\nComponent Loadings (PC1 and PC2):")
loadings_df = pd.DataFrame(
    pca.components_[:2].T,
    columns=['PC1', 'PC2'],
    index=pca_vars
)
print(loadings_df)

In [ ]:
# Visualize PCA results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Scree plot
ax = axes[0, 0]
ax.plot(range(1, len(pca.explained_variance_ratio_)+1), 
        pca.explained_variance_ratio_, 'o-', linewidth=2, markersize=10)
ax.set_xlabel('Principal Component', fontsize=11, fontweight='bold')
ax.set_ylabel('Explained Variance Ratio', fontsize=11, fontweight='bold')
ax.set_title('Scree Plot', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# Plot 2: Cumulative variance
ax = axes[0, 1]
ax.plot(range(1, len(cumsum)+1), cumsum, 'o-', linewidth=2, markersize=10, color='coral')
ax.axhline(0.8, color='red', linestyle='--', label='80% threshold')
ax.axhline(0.9, color='orange', linestyle='--', label='90% threshold')
ax.set_xlabel('Number of Components', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative Variance Explained', fontsize=11, fontweight='bold')
ax.set_title('Cumulative Variance Explained', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

# Plot 3: Component loadings heatmap
ax = axes[1, 0]
sns.heatmap(loadings_df, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, cbar_kws={'label': 'Loading'})
ax.set_title('Component Loadings (PC1 & PC2)', fontsize=12, fontweight='bold')
ax.set_ylabel('Variable', fontsize=11, fontweight='bold')

# Plot 4: Biplot (PC1 vs PC2)
ax = axes[1, 1]
# Plot samples
ax.scatter(pca_components[:, 0], pca_components[:, 1], alpha=0.3, s=10)

# Plot variable vectors
for i, var in enumerate(pca_vars):
    ax.arrow(0, 0, pca.components_[0, i]*3, pca.components_[1, i]*3,
            head_width=0.1, head_length=0.1, fc='red', ec='red', linewidth=2)
    ax.text(pca.components_[0, i]*3.5, pca.components_[1, i]*3.5, var,
           fontsize=10, fontweight='bold', ha='center')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', 
             fontsize=11, fontweight='bold')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', 
             fontsize=11, fontweight='bold')
ax.set_title('PCA Biplot', fontsize=12, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.axvline(0, color='black', linewidth=0.5, linestyle='--')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/multivariate_statistical_tests/03_pca_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 03_pca_analysis.png")

## 6. Summary and Key Findings

In [ ]:
print("="*80)
print("KEY FINDINGS: MULTIVARIATE STATISTICAL ANALYSIS")
print("="*80)

print("\n1. CORRELATION STRENGTH (with mortality):")
print("   Pearson correlations:")
for var_name in core_variables.keys():
    corr = pearson_matrix.loc[core_variables[var_name], mortality_var]
    print(f"     {var_name}: r = {corr:.4f}")

print("\n2. PARTIAL CORRELATIONS (controlling for confounders):")
for _, row in partial_df.iterrows():
    print(f"   {row['Variable']}:")
    print(f"     Simple: r = {row['Simple_r']:.4f}")
    print(f"     Partial: r = {row['Partial_r']:.4f} (change: {row['Difference']:+.4f})")

print("\n3. MULTIPLE REGRESSION:")
print(f"   Additive model R²: {adj_r2_model1:.4f}")
print(f"   Interaction model R²: {adj_r2_model2:.4f}")
print(f"   Improvement: {adj_r2_model2 - adj_r2_model1:.4f}")
print(f"   Interactions {'significantly' if p_value < 0.05 else 'not significantly'} improve model (p = {p_value:.4f})")

print("\n4. MULTICOLLINEARITY (VIF):")
for _, row in vif_data.iterrows():
    status = 'Low' if row['VIF'] < 5 else 'Moderate' if row['VIF'] < 10 else 'High'
    print(f"   {row['Variable']}: VIF = {row['VIF']:.2f} ({status})")

print("\n5. PRINCIPAL COMPONENT ANALYSIS:")
print(f"   PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance")
print(f"   First 2 PCs explain {cumsum[1]*100:.1f}% of variance")
print(f"   PC1 loadings (strongest to weakest):")
pc1_loadings = loadings_df['PC1'].abs().sort_values(ascending=False)
for var in pc1_loadings.index:
    print(f"     {var}: {loadings_df.loc[var, 'PC1']:.3f}")

print("\n" + "="*80)
print("CONCLUSIONS")
print("="*80)
print("")
print("1. Daytime heat and nighttime recovery are strongly correlated with mortality")
print("2. Partial correlations reveal independent effects after controlling for confounders")
print("3. Interaction effects are statistically significant - heat stress is synergistic")
print("4. Multicollinearity is moderate - variables measure related but distinct phenomena")
print("5. PCA reveals a dominant heat stress factor explaining most variance")
print("")
print("Recommendation: Use multivariate models that account for interactions")
print("="*80)

## 7. Export Results

In [ ]:
# Save correlation matrices
pearson_matrix.to_csv(CATTLE_DATA_DIR / 'multivariate_pearson_correlations.csv')
spearman_matrix.to_csv(CATTLE_DATA_DIR / 'multivariate_spearman_correlations.csv')
kendall_matrix.to_csv(CATTLE_DATA_DIR / 'multivariate_kendall_correlations.csv')
print("Correlation matrices saved")

# Save partial correlations
partial_df.to_csv(CATTLE_DATA_DIR / 'partial_correlations.csv', index=False)
print("Partial correlations saved")

# Save PCA results
pca_results = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
    'Explained_Variance': pca.explained_variance_ratio_,
    'Cumulative_Variance': cumsum
})
pca_results.to_csv(CATTLE_DATA_DIR / 'pca_results.csv', index=False)
loadings_df.to_csv(CATTLE_DATA_DIR / 'pca_loadings.csv')
print("PCA results saved")

# Save VIF
vif_data.to_csv(CATTLE_DATA_DIR / 'variance_inflation_factors.csv', index=False)
print("VIF data saved")

print("\n✓ Analysis complete! All figures saved to figures/multivariate_statistical_tests/")